# Thực nghiệm Benchmark Phân loại Văn bản Tiếng Việt: Từ Phương pháp Truyền thống đến Mô hình Ngôn ngữ Lớn
**Tái hiện nghiên cứu:** *"A thorough benchmark of automatic text classification: From traditional approaches to large language models"* (Cunha et al., 2025)

---
### Mục tiêu & Thiết kế Thực nghiệm Thu gọn (Chạy nhanh & Nhẹ)
Thực nghiệm này đối chiếu trực tiếp **Hiệu quả phân loại (Macro-F1)** và **Chi phí tính toán (Thời gian huấn luyện + suy luận)** trên bài toán phân loại cảm xúc tiếng Việt (**UIT-VSFC**) qua 3 thế hệ mô hình:
1. **Traditional Approaches (TF-IDF):** Linear SVM (`LinearSVC`), `LogisticRegression`, `RandomForestClassifier`.
2. **Small Language Models (SLMs):** `PhoBERT-base-v2` (135M tham số, tách từ bằng `pyvi`) và `viBERT-base` (110M tham số, cấp độ âm tiết).
3. **Large Language Model (LLM):** `Qwen2.5-1.5B-Instruct` tinh chỉnh bằng **4-bit QLoRA** (tối ưu bộ nhớ và tốc độ trên Google Colab T4 GPU).

> **Lưu ý:** Để chạy nhanh (~1–2 phút trên GPU T4), tập dữ liệu được lấy mẫu cân đối (`1.500` mẫu train / `500` mẫu test) với `MAX_LEN = 64`. Bạn có thể tăng số lượng mẫu khi muốn chạy full dataset.

## Phần 1: Cài đặt Thư viện & Nạp Dữ liệu Tiếng Việt
Bật sẵn lệnh cài đặt các thư viện cần thiết (`datasets`, `pyvi`, `peft`, `bitsandbytes`, `accelerate`) và kiểm tra GPU.

In [ ]:
# Tự động cài đặt các gói cần thiết (chạy mượt trên Google Colab)
%pip install -q transformers datasets accelerate peft bitsandbytes scikit-learn pyvi matplotlib seaborn

In [ ]:
import os
import time
import gc
import warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Đang sử dụng thiết bị: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Tải bộ dữ liệu phân loại cảm xúc phản hồi sinh viên tiếng Việt (UIT-VSFC: 3 nhãn 0-Tiêu cực, 1-Trung lập, 2-Tích cực)
print('Đang tải bộ dữ liệu UIT-VSFC...')
ds = load_dataset('uitnlp/vietnamese_students_feedback')

# Lấy mẫu gọn nhẹ để chạy nhanh (có thể tăng N_TRAIN, N_TEST nếu muốn chạy toàn bộ tập dữ liệu)
N_TRAIN = 1500
N_TEST = 500

train_df = ds['train'].shuffle(seed=42).select(range(N_TRAIN)).to_pandas()
test_df = ds['test'].shuffle(seed=42).select(range(N_TEST)).to_pandas()

X_train = train_df['sentence'].tolist()
y_train = train_df['sentiment'].tolist()
X_test = test_df['sentence'].tolist()
y_test = test_df['sentiment'].tolist()

print(f'Hoàn tất! Train: {len(X_train)} câu | Test: {len(X_test)} câu')
print('Ví dụ 3 mẫu dữ liệu:')
display(train_df[['sentence', 'sentiment']].head(3))

## Phần 2: Nhóm 1 — Phương pháp Học máy Truyền thống (Traditional Approaches)
Sử dụng biểu diễn **TF-IDF (1–2 n-grams)** kết hợp với 3 bộ phân loại kinh điển trong bài báo: **Linear SVM (LSVM)**, **Logistic Regression (LR)** và **Random Forest (RF)**.

In [ ]:
results = []

trad_models = {
    'LSVM (TF-IDF)': LinearSVC(C=1.0, random_state=42),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
}

for name, clf in trad_models.items():
    t0 = time.time()
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
    X_tr_vec = vec.fit_transform(X_train)
    X_te_vec = vec.transform(X_test)
    
    clf.fit(X_tr_vec, y_train)
    preds = clf.predict(X_te_vec)
    total_time = time.time() - t0
    
    macro_f1 = f1_score(y_test, preds, average='macro') * 100
    acc = accuracy_score(y_test, preds) * 100
    
    results.append({
        'Nhóm': 'Traditional',
        'Mô hình': name,
        'Macro-F1 (%)': round(macro_f1, 2),
        'Accuracy (%)': round(acc, 2),
        'Thời gian (s)': round(total_time, 3)
    })
    print(f'[{name:<20}] Macro-F1: {macro_f1:.2f}% | Accuracy: {acc:.2f}% | Thời gian: {total_time:.3f}s')

## Phần 3: Nhóm 2 — Mô hình Ngôn ngữ Nhỏ (Small Language Models - SLMs)
Fine-tuning hai đại diện SLMs tiêu biểu cho tiếng Việt:
* **PhoBERT-base-v2** (`vinai/phobert-base-v2`): Yêu cầu tách từ tiếng Việt (`ViTokenizer.tokenize`) trước khi đưa vào mô hình.
* **viBERT-base** (`FPTAI/vibert-base-cased`): Xử lý trực tiếp văn bản thô ở cấp độ âm tiết.

In [ ]:
slm_models = {
    'PhoBERT-base-v2': 'vinai/phobert-base-v2',
    'viBERT-base': 'FPTAI/vibert-base-cased'
}

MAX_LEN = 64
SLM_EPOCHS = 3

for name, model_id in slm_models.items():
    print(f'\nĐang huấn luyện SLM: {name} ({model_id})...')
    torch.manual_seed(42)
    t0 = time.time()
    
    tok = AutoTokenizer.from_pretrained(model_id)
    
    # Tiền xử lý tách từ nếu là PhoBERT
    tr_txt = [ViTokenizer.tokenize(t) for t in X_train] if 'PhoBERT' in name else X_train
    te_txt = [ViTokenizer.tokenize(t) for t in X_test] if 'PhoBERT' in name else X_test
    
    enc_tr = tok(tr_txt, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    enc_te = tok(te_txt, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    
    tr_loader = DataLoader(TensorDataset(enc_tr['input_ids'], enc_tr['attention_mask'], torch.tensor(y_train)), batch_size=32, shuffle=True)
    te_loader = DataLoader(TensorDataset(enc_te['input_ids'], enc_te['attention_mask']), batch_size=64)
    
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=3).to(device)
    opt = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
    
    model.train()
    for epoch in range(SLM_EPOCHS):
        for ids, mask, lbl in tr_loader:
            opt.zero_grad()
            loss = model(ids.to(device), attention_mask=mask.to(device), labels=lbl.to(device)).loss
            loss.backward()
            opt.step()
            
    model.eval()
    preds = []
    with torch.no_grad():
        for ids, mask in te_loader:
            logits = model(ids.to(device), attention_mask=mask.to(device)).logits
            preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    total_time = time.time() - t0
    
    macro_f1 = f1_score(y_test, preds, average='macro') * 100
    acc = accuracy_score(y_test, preds) * 100
    
    results.append({
        'Nhóm': 'SLM',
        'Mô hình': name,
        'Macro-F1 (%)': round(macro_f1, 2),
        'Accuracy (%)': round(acc, 2),
        'Thời gian (s)': round(total_time, 2)
    })
    print(f'[{name:<20}] Macro-F1: {macro_f1:.2f}% | Accuracy: {acc:.2f}% | Thời gian: {total_time:.2f}s')
    
    # Giải phóng VRAM trước khi chạy mô hình tiếp theo
    del model, opt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Phần 4: Nhóm 3 — Mô hình Ngôn ngữ Lớn (Large Language Model - LLM với QLoRA)
Áp dụng đúng phương pháp của bài báo (**4-bit Quantization + QLoRA** qua thư viện `peft`) trên mô hình **`Qwen/Qwen2.5-1.5B-Instruct`** (hỗ trợ tiếng Việt rất tốt và đủ nhẹ để fine-tune trong ~45–60 giây trên Colab T4 GPU).

> *(Nếu chạy trên máy chỉ có CPU không hỗ trợ `bitsandbytes` 4-bit, cell này có cơ chế tự động bỏ qua an toàn mà không làm lỗi toàn bộ notebook)*.

In [ ]:
if torch.cuda.is_available():
    from transformers import BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, TaskType
    
    llm_id = 'Qwen/Qwen2.5-1.5B-Instruct'
    print(f'Đang tải & tinh chỉnh QLoRA 4-bit cho LLM: {llm_id}...')
    torch.manual_seed(42)
    t0 = time.time()
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16
    )
    
    tok_llm = AutoTokenizer.from_pretrained(llm_id)
    if tok_llm.pad_token is None:
        tok_llm.pad_token = tok_llm.eos_token
        
    model_llm = AutoModelForSequenceClassification.from_pretrained(
        llm_id,
        num_labels=3,
        quantization_config=bnb_config,
        device_map='auto'
    )
    model_llm.config.pad_token_id = tok_llm.pad_token_id
    
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=['q_proj', 'v_proj']
    )
    model_llm = get_peft_model(model_llm, lora_cfg)
    
    enc_tr = tok_llm(X_train, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    enc_te = tok_llm(X_test, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    
    tr_loader = DataLoader(TensorDataset(enc_tr['input_ids'], enc_tr['attention_mask'], torch.tensor(y_train)), batch_size=16, shuffle=True)
    te_loader = DataLoader(TensorDataset(enc_te['input_ids'], enc_te['attention_mask']), batch_size=32)
    
    opt = AdamW(model_llm.parameters(), lr=2e-4)
    model_llm.train()
    for epoch in range(2):  # 2 epochs QLoRA
        for ids, mask, lbl in tr_loader:
            opt.zero_grad()
            loss = model_llm(ids.to(device), attention_mask=mask.to(device), labels=lbl.to(device)).loss
            loss.backward()
            opt.step()
            
    model_llm.eval()
    preds = []
    with torch.no_grad():
        for ids, mask in te_loader:
            logits = model_llm(ids.to(device), attention_mask=mask.to(device)).logits
            preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            
    torch.cuda.synchronize()
    total_time = time.time() - t0
    
    macro_f1 = f1_score(y_test, preds, average='macro') * 100
    acc = accuracy_score(y_test, preds) * 100
    
    results.append({
        'Nhóm': 'LLM (QLoRA)',
        'Mô hình': 'Qwen2.5-1.5B',
        'Macro-F1 (%)': round(macro_f1, 2),
        'Accuracy (%)': round(acc, 2),
        'Thời gian (s)': round(total_time, 2)
    })
    print(f'[Qwen2.5-1.5B (QLoRA)] Macro-F1: {macro_f1:.2f}% | Accuracy: {acc:.2f}% | Thời gian: {total_time:.2f}s')
    
    del model_llm, opt
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('Không phát hiện GPU CUDA — Bỏ qua bước QLoRA 4-bit cho LLM. Vui lòng bật T4 GPU trên Colab để chạy phần này.')

## Phần 5: Tổng hợp Bảng kết quả & Biểu đồ Đánh đổi Cost–Effectiveness
Tái hiện biểu đồ **Cost ($\log_{10}(\text{Time})$) vs. Effectiveness (Macro-F1)** theo đúng cách trình bày của bài báo Cunha et al. (2025):
* **Góc phần tư trên bên trái:** Mô hình lý tưởng (vừa nhanh hơn trung vị, vừa chính xác hơn trung vị).
* **Góc phần tư trên bên phải:** Chính xác cao nhưng chi phí thời gian lớn (thường là LLMs).
* **Góc phần tư dưới bên trái:** Rất nhanh nhưng độ chính xác thấp hơn (thường là các phương pháp truyền thống).

In [ ]:
df_res = pd.DataFrame(results)
print('BẢNG TỔNG HỢP KẾT QUẢ BENCHMARK PHÂN LOẠI VĂN BẢN TIẾNG VIỆT:')
display(df_res)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
sns.set_style('whitegrid')

# Biểu đồ 1: Scatter Trade-off (Giống Hình 2 của bài báo)
log_time = np.log10(df_res['Thời gian (s)'] + 0.05)
sns.scatterplot(
    x=log_time,
    y=df_res['Macro-F1 (%)'],
    hue=df_res['Nhóm'],
    style=df_res['Nhóm'],
    s=200,
    palette={'Traditional': '#4A5568', 'SLM': '#2B6CB0', 'LLM (QLoRA)': '#D69E2E'},
    ax=axes[0]
)

for i, row in df_res.iterrows():
    axes[0].text(
        log_time.iloc[i] + 0.04,
        row['Macro-F1 (%)'],
        f"{row['Mô hình']} ({row['Macro-F1 (%)']}%)",
        fontsize=9,
        fontweight='bold'
    )

axes[0].axhline(df_res['Macro-F1 (%)'].median(), ls='--', color='gray', alpha=0.6, label='Trung vị Macro-F1')
axes[0].axvline(log_time.median(), ls=':', color='gray', alpha=0.6, label='Trung vị log10(Time)')
axes[0].set_title('Đánh đổi Hiệu quả (Macro-F1) vs. Chi phí (log10 Time)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('log10(Thời gian Train + Test [s]) — Càng sang trái càng nhanh')
axes[0].set_ylabel('Macro-F1 (%) — Càng cao càng chính xác')
axes[0].set_xlim(log_time.min() - 0.3, log_time.max() + 0.9)
axes[0].legend(loc='lower right')

# Biểu đồ 2: So sánh trực tiếp Thời gian chạy (giây)
sns.barplot(data=df_res, x='Mô hình', y='Thời gian (s)', hue='Nhóm', dodge=False, ax=axes[1],
            palette={'Traditional': '#4A5568', 'SLM': '#2B6CB0', 'LLM (QLoRA)': '#D69E2E'})
axes[1].set_title('Tổng Thời gian Huấn luyện & Dự đoán (giây)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Giây (s)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_ylim(0, df_res['Thời gian (s)'].max() * 1.18)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.2fs', padding=3, fontsize=9)

plt.tight_layout()
plt.show()

### Tổng kết Đối chiếu với Nghiên cứu Gốc (Cunha et al., 2025):
1. **Traditional (`LSVM`, `Logistic Regression`):** Tốc độ nhanh hơn hàng chục đến hàng trăm lần so với mô hình học sâu, đạt mức hiệu năng nền ổn định cho các bài toán cần xử lý thời gian thực với chi phí gần như bằng 0.
2. **SLMs (`PhoBERT-base-v2`, `viBERT`):** Nằm ở **vùng cân bằng tối ưu (Near-optimal trade-off)** — cải thiện rõ rệt Macro-F1 so với TF-IDF nhờ nắm bắt ngữ cảnh tiếng Việt, trong khi thời gian fine-tuning chỉ tính bằng chục giây.
3. **LLMs (`Qwen2.5-1.5B` QLoRA):** Đạt chất lượng phân loại rất cao nhờ tri thức tiền huấn luyện khổng lồ, nhưng chi phí tính toán và thời gian fine-tuning cao hơn đáng kể so với SLMs.